<a href="https://colab.research.google.com/github/melissa-04/melisayla-biyoinformatik/blob/main/notebooks/rna-seq/09_zenginlestirme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zenginleştirme: 1.385 genin ortak süreçleri

Sekizinci rehber bize bir liste bıraktı: yüzlerce yükselen, yüzlerce düşen gen. Tek tek okumak imkânsız; doğru soru şu: bu genler rastgele bir kalabalık mı, yoksa belli biyolojik süreçlerin takımları hâlinde mi hareket ediyorlar? Bu defterde bu soruyu soran testi (over-representation analysis, ORA) hazır araç kullanmadan, hipergeometrik dağılımla kendi elimizle kuracağız.

Model yeniden uyacağı için ilk hücreler iki-üç dakika sürer.

In [1]:
%pip install -q pydeseq2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.8/188.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==

## 1. Özet tekrar

Sekizinci rehberin hattını tek hücrede yeniden koşup üç şey çıkarıyoruz: test edilen genlerin tamamı (evren), anlamlı düşenler ve anlamlı yükselenler. Katalog gen adlarıyla konuştuğu için bu defterde ada geçiyoruz; yaklaşık yirmi adın birden fazla kimliğe denk gelmesi bu ölçekte sonucu değiştirmez, ama altıncı rehberin uyarısını unutmuyoruz.

In [2]:
import pandas as pd
import numpy as np
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

t = pd.read_csv('https://raw.githubusercontent.com/melissa-04/melisayla-biyoinformatik/main/data/rna-seq/sayim_tablosu.csv')
ornekler = ['WT_1', 'WT_2', 'WT_3', 'KO_1', 'KO_2', 'KO_3']
g = t.set_index('gen_id')[ornekler]
sayim = g.T
sayim = sayim[sayim.columns[sayim.sum(axis=0) >= 10]]
meta = pd.DataFrame({'genotip': ['WT'] * 3 + ['KO'] * 3}, index=sayim.index)

dds = DeseqDataSet(counts=sayim, metadata=meta,
                   design="~genotip", ref_level=["genotip", "WT"], quiet=True)
dds.deseq2()
st = DeseqStats(dds, contrast=["genotip", "KO", "WT"], quiet=True)
st.summary()
sonuc = st.results_df
sonuc['gen_adi'] = t.set_index('gen_id')['gen_adi'].reindex(sonuc.index)

evren = sonuc.dropna(subset=['padj'])
evren_adlar = set(evren.gen_adi)
anlamli = evren[(evren.padj < 0.05) & (evren.log2FoldChange.abs() > 1)]
dusen = set(anlamli[anlamli.log2FoldChange < 0].gen_adi)
yukselen = set(anlamli[anlamli.log2FoldChange > 0].gen_adi)
print('evren:', len(evren_adlar), '| düşen:', len(dusen), '| yükselen:', len(yukselen))

/tmp/ipykernel_3149/1939940024.py:13: DeprecationWarning: ref_level is deprecated and no longer has any effect. It will beremoved in a future release.
  dds = DeseqDataSet(counts=sayim, metadata=meta,


evren: 17238 | düşen: 754 | yükselen: 631


## 2. Fikir: dört sayı

Testin bütün malzemesi dört sayıdır. **N**: evrendeki gen sayısı — burada evren, DESeq2'nin gerçekten test ettiği genlerdir, bütün genom değil. **n**: listemizin boyu. **K**: sorguladığımız gen setinin evrende bulunan üyeleri. **k**: setin listeyle örtüşmesi. Şans beklentisi n·K/N'dir; örneğin 753 genlik düşen liste, 16.814'lük evren ve 11 genlik bir set için beklenti 0,5 gen civarıdır. Gözlenen örtüşme bunun çok üstündeyse, hipergeometrik dağılım "bu kadar örtüşme şansla ne sıklıkla olur" sorusuna kesin bir p değeri verir.

Önce katalog. Aşağıdaki on iki set benim derlemem: bu deneyin biyolojisine yakın süreçler artı birkaç ilgisiz kontrol. Gerçek analizde binlerce setlik hazır kataloglar (GO, MSigDB, .gmt dosyaları) kullanılır; mekanizma birebir aynıdır.

In [3]:
setler = {
 'hem_biyosentezi': ['Alas2','Alad','Hmbs','Uros','Urod','Cpox','Ppox','Fech','Slc25a38'],
 'demir_metabolizmasi': ['Slc25a37','Tfrc','Slc11a2','Steap3','Fth1','Ftl1','Ireb2','Aco1','Slc40a1','Tfr2'],
 'eritrosit_zar_iskeleti': ['Slc4a1','Spta1','Sptb','Ank1','Epb41','Epb42','Dmtn','Add2','Gypa','Gypc','Rhag'],
 'globinler_yetiskin': ['Hba-a1','Hba-a2','Hbb-bs','Hbb-bt'],
 'globinler_embriyonik': ['Hbb-y','Hbb-bh1','Hba-x'],
 'eritroid_tf': ['Gata1','Klf1','Tal1','Nfe2','Zfpm1','Lmo2','Gfi1b','Epor'],
 'mitokondriyal_solunum': ['mt-Co1','mt-Co2','mt-Co3','mt-Nd1','mt-Nd2','mt-Nd4','mt-Cytb','mt-Atp6'],
 'translasyon': ['Eef1a1','Eef2','Eif4a1','Rpl4','Rpl6','Rps3','Rps6','Rpl13a','Rps19','Rpl11'],
 'hepatosit_programi': ['Alb','Afp','Apoa1','Apoa2','Apob','Ttr','Ahsg','Fga','Fgb','Fgg'],
 'hucre_dongusu': ['Mki67','Ccnb1','Ccna2','Cdk1','Top2a','Plk1','Bub1','Aurkb','Mcm2','Pcna'],
 'apoptoz': ['Bax','Bak1','Casp3','Casp9','Bcl2','Bcl2l1','Bid','Apaf1','Trp53','Bbc3'],
 'otofaji': ['Atg5','Atg7','Map1lc3b','Becn1','Sqstm1','Ulk1','Atg3','Atg12'],
}
print(len(setler), 'set,', sum(len(v) for v in setler.values()), 'gen')

12 set, 101 gen


## 3. Test: hipergeometrik + çoklu test

Sekizinci rehberin dersi burada da geçerli: on iki set birden test ediyoruz, dolayısıyla p değerlerine Benjamini-Hochberg düzeltmesi uyguluyoruz. Önce düşenler.

In [4]:
from scipy.stats import hypergeom

def ora(liste, N_adlar):
    N = len(N_adlar)
    n = len(liste & N_adlar)
    satirlar = []
    for ad, gl in setler.items():
        S = set(gl) & N_adlar
        K = len(S)
        k = len(S & liste)
        p = hypergeom.sf(k - 1, N, K, n) if k > 0 else 1.0
        satirlar.append((ad, k, K, p))
    df = pd.DataFrame(satirlar, columns=['set', 'örtüşme', 'set_boyu', 'p']).sort_values('p').reset_index(drop=True)
    m = len(df)
    df['padj'] = np.minimum.accumulate((df.p * m / (df.index + 1))[::-1])[::-1]
    return df

dusen_ora = ora(dusen, evren_adlar)
print(dusen_ora.to_string(index=False, float_format=lambda x: f'{x:.2e}'))

                   set  örtüşme  set_boyu        p     padj
eritrosit_zar_iskeleti        6        11 2.63e-06 3.15e-05
       hem_biyosentezi        5         9 1.72e-05 1.03e-04
           eritroid_tf        3         8 3.96e-03 1.58e-02
   demir_metabolizmasi        3        10 7.94e-03 2.38e-02
    globinler_yetiskin        2         4 1.08e-02 2.59e-02
               apoptoz        1        10 3.61e-01 5.41e-01
         hucre_dongusu        1        10 3.61e-01 5.41e-01
    hepatosit_programi        1        10 3.61e-01 5.41e-01
           translasyon        0        10 1.00e+00 1.00e+00
 mitokondriyal_solunum        0         8 1.00e+00 1.00e+00
  globinler_embriyonik        0         3 1.00e+00 1.00e+00
               otofaji        0         8 1.00e+00 1.00e+00


## 4. Okumak

Tabloyu iki yönden okuyun. Tepedeki setler deneyin hikâyesini anlatır. Dipteki 1,0'lar ise testin dürüstlüğünü: ilgisiz kontroller (otofaji, translasyon, hücre döngüsü) zenginleşmiyorsa test her şeye "anlamlı" demiyor demektir. Negatif kontrolsüz zenginleştirme tablosu okunmaz.

Bir sınır daha: zenginleşme, mekanizma kanıtı değildir. ORA yalnız "bu takım listede şans eseri olamayacak kadar kalabalık" der; kimin kimi düşürdüğünü söylemez.

## 5. Evren tuzağı

En yaygın hata, evren olarak bütün genomu almaktır. Deneyle görelim: aynı düşen listeyi, aynı seti, bir de tablodaki 56.065 genin tamamına karşı test edin. Fetal karaciğerde hiç ifade edilmeyen on binlerce gen bu listeye zaten giremezdi; onları paydaya koymak örtüşmeyi olduğundan nadir gösterir ve p değerini yapay biçimde küçültür.

In [5]:
tablo_adlari = set(t.gen_adi)
yanlis = ora(dusen, tablo_adlari)
karsilastir = dusen_ora.set_index('set')[['p']].join(
    yanlis.set_index('set')[['p']], lsuffix='_dogru_evren', rsuffix='_yanlis_evren')
print(karsilastir.head(5).to_string(float_format=lambda x: f'{x:.2e}'))

                        p_dogru_evren  p_yanlis_evren
set                                                  
eritrosit_zar_iskeleti       2.63e-06        2.59e-09
hem_biyosentezi              1.72e-05        5.34e-08
eritroid_tf                  3.96e-03        1.31e-04
demir_metabolizmasi          7.94e-03        2.74e-04
globinler_yetiskin           1.08e-02        1.07e-03


## 6. Tekrarlanabilirlik notu

Bu serinin sekizinci rehberinde aynı veri ve aynı kod, iki farklı pydeseq2 sürümünde bir genin sırasını 14'ten 1'e taşıdı: sürümler tek bir genin dispersiyonunu farklı kestirince p değeri 10⁻⁶⁷'den 10⁻¹⁴⁵'e kaydı. Ders iki cümle: analiz raporuna araç sürümleri yazılır, ve bu mertebedeki p değerleri arasında tekil sıralamaya değil kümeye güvenilir. Sürümünüzü kaydetme alışkanlığı tek satırdır:

In [6]:
import pydeseq2, scipy
print('pydeseq2', pydeseq2.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__)

pydeseq2 0.5.4 | pandas 3.0.5 | scipy 1.16.3


## Kendin dene

Üç görev. Birincisi: düşenler tablonuzdan en güçlü zenginleşen setin adını, örtüşme/set boyu sayılarını ve padj değerini not edin. İkincisi: evren deneyinden aynı setin iki p değerini yan yana yazın ve tek cümleyle açıklayın: yanlış evren p'yi hangi yöne, neden oynatıyor? Üçüncüsü: `ora(yukselen, evren_adlar)` ile yükselenleri test edin; tepeye çıkan seti değerleriyle not edin ve tek cümleyle söyleyin: bu sonuç, sekizinci rehberde gördüğünüz hangi bulguyla aynı hikâyeyi anlatıyor? Not: bu set 0,05 çizgisinin hemen kıyısında; sürüm farkı sizde çizginin öbür yanına düşürebilir — düşerse bu da başlı başına bir gözlemdir, onu da yazın.